# BTP-1: numbered GPU experiment
Run cells in order. This trains the proposed **96 x 96 x 96, width-16** model on all 369 labelled BraTS 2020 patients with separate training, validation and test partitions.

Before starting: upload `btp1_gpu_code.zip` to Google Drive in `MyDrive/BTP1_GPU/`. Open this notebook in Google Colab and select a GPU runtime. The download uses the existing project's Kaggle mirror helper; comply with the dataset's access terms. You can instead use your existing BraTS data as described in Step 4.

Steps 1–10 are the main experiment. Steps 11–12 are additional experiments. Step 13 checks and downloads the results. Stop and inspect any failed cell before continuing. No GPU training has been run by creating this notebook.

## Archived method and output protection
The completed GPU results used effectively unweighted cross-entropy: configured class weights cancel under mean reduction at physical batch size one. The underlying archived training code is retained for reproduction; correcting the loss requires a separately labelled experiment. The final partial accumulation window is also scaled by the configured factor of four.

Fresh runs below write into a timestamped directory, leaving the archived `roi96_seed42` folder intact. This template has not been executed as part of the report review.


## 1. Confirm GPU

In [ ]:
import sys, subprocess, shutil, json, os, time
from pathlib import Path
import torch
subprocess.run(['nvidia-smi'], check=True)
assert torch.cuda.is_available(), 'Select a GPU runtime, then rerun this cell.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)


## 2. Mount Drive and unpack the code
The ZIP is small and contains source code and labels, not MRI volumes. Outputs are saved to Drive; MRI and cached tensors stay on the runtime disk for faster access.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import zipfile
DRIVE = Path('/content/drive/MyDrive/BTP1_GPU')
BUNDLE = DRIVE / 'btp1_gpu_code.zip'
assert BUNDLE.is_file(), f'Upload the code ZIP here first: {BUNDLE}'
PROJECT = Path('/content/BTP1_GPU')
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    z.extractall(PROJECT)
os.chdir(PROJECT)
RUN_ID = time.strftime('roi96_seed42_%Y%m%d_%H%M%S')
ARTIFACTS = DRIVE / 'runs' / RUN_ID
ARTIFACTS.mkdir(parents=True, exist_ok=True)
DATA_ROOT = Path('/content/brats2020_training')
MAPPING = PROJECT / 'labels' / 'name_mapping.csv'
CACHE = Path('/content/roi96_cache')
RUN = ARTIFACTS / 'fixed_split'


## 3. Install dependencies and record the environment

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT), 'kagglehub'], check=True)
def run(*args):
    command = [sys.executable, '-u', *map(str, args)]
    print('Running:', ' '.join(command), flush=True)
    started = time.monotonic()
    with subprocess.Popen(command, cwd=PROJECT, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True,
                          errors='replace', bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        returncode = process.wait()
    if returncode:
        raise RuntimeError(f'Command failed with exit code {returncode}. Read the downloader/script error printed above.')
    print(f'Finished in {(time.monotonic()-started)/60:.1f} minutes', flush=True)
environment = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(ARTIFACTS / 'environment.txt').write_text(environment)
shutil.copy2(BUNDLE, ARTIFACTS / 'source_code.zip')
(ARTIFACTS / 'gpu.txt').write_text(subprocess.check_output(['nvidia-smi'], text=True))
import pandas as pd


## 4. Obtain the real dataset
Default: download the 1,845 training NIfTI files using the supplied helper and the 369-patient grade mapping. Colab's KaggleHub cache service is bypassed to avoid HTTP 500 mount/support errors and to keep files under DATA_ROOT. Existing readable NIfTI files are skipped when retrying. The uncompressed cohort is about 30.83 GiB; allow extra disk space for the download and processed caches. This is the network-dependent part. If Kaggle requests authorization, complete its normal sign-in/access steps before retrying; the helper cannot grant access.

**Alternative:** upload/extract your existing `data/raw` cohort into the Colab runtime, set `DATA_ROOT` to that directory, and skip the download command below. Use the original four modalities and segmentation masks, not the existing 32-cubed caches. The helper handles the mirror's special segmentation filename for patient 355.

In [ ]:
print('Local free disk GiB:', round(shutil.disk_usage('/content').free / 2**30, 1))
os.environ['DISABLE_COLAB_CACHE'] = 'true'
assert len(pd.read_csv(MAPPING)) == 369
run('scripts/download_brats2020.py', '--mapping-csv', MAPPING,
    '--output-dir', DATA_ROOT, '--workers', 2)


## 5. Build patient splits and audit the data
Expected: 369 patients, with 257 training, 56 validation and 56 test patients. Five-fold IDs are also prepared. Do not continue if the audit fails.

In [ ]:
MANIFEST = ARTIFACTS / 'manifest.csv'
run('-m', 'btp3d.manifest', '--data-root', DATA_ROOT, '--mapping-csv', MAPPING,
    '--output', MANIFEST, '--seed', 42, '--folds', 5, '--strict',
    '--audit-output', ARTIFACTS / 'data_audit.csv')
manifest = pd.read_csv(MANIFEST)
assert len(manifest) == manifest.patient_id.nunique() == 369
assert manifest.label_name.value_counts().to_dict() == {'HGG': 293, 'LGG': 76}
audit = pd.read_csv(ARTIFACTS / 'data_audit.csv')
assert len(audit) == 369 and audit.valid.all()
display(manifest.groupby(['split', 'label_name']).size().unstack(fill_value=0))


## 6. Preprocess at 96-cubed resolution
This cell uses CPU for NIfTI processing. GPU acceleration starts during model training. The mask defines the tumour crop and is not an input channel.

In [ ]:
run('-m', 'btp3d.preprocess', '--manifest', MANIFEST, '--output-dir', CACHE,
    '--spatial-size', 96, 96, 96, '--crop-mode', 'roi', '--margin', 16, '--workers', 2)
CACHED = CACHE / 'manifest_cached.csv'
cached = pd.read_csv(CACHED)
assert len(cached) == 369 and cached.cache_path.map(lambda p: Path(p).is_file()).all()
shutil.copy2(CACHED, ARTIFACTS / 'manifest_cached.csv')
print('Cached patients:', len(cached))


## 7. Inspect preprocessing examples
Inspect each image for recognisable brain anatomy, matching modality locations and a sensible tumour crop before training.

In [ ]:
run('-m', 'btp3d.qa', '--manifest', CACHED, '--output-dir', ARTIFACTS / 'qa', '--per-split', 2)
from IPython.display import display
from PIL import Image
for path in sorted((ARTIFACTS / 'qa').glob('*.png')):
    print(path.name)
    display(Image.open(path))


## 8. Check learning on training patients only
The eight-patient check is for debugging. It uses width 4 with dropout disabled and is not the main experiment or a performance result. Expected: 100% training accuracy, loss below 0.02 and matching predictions after checkpoint reload.

In [ ]:
DEBUG_MANIFEST = ARTIFACTS / 'training_only_debug_manifest.csv'
cached.loc[cached.split == 'train'].to_csv(DEBUG_MANIFEST, index=False)
run('-m', 'btp3d.overfit_check', '--manifest', DEBUG_MANIFEST,
    '--output-dir', ARTIFACTS / 'overfit_check', '--samples-per-class', 4,
    '--epochs', 200, '--base-channels', 4, '--seed', 42)
debug_result = json.loads((ARTIFACTS / 'overfit_check' / 'overfit_check.json').read_text())
assert debug_result['passed']
print(json.dumps(debug_result, indent=2))


## 9. Train the main GPU model
This is the required main experiment: width 16, input 96-cubed, physical batch 1, accumulation 4, maximum 60 epochs, early-stopping patience 12. Checkpoints are selected by validation balanced accuracy. The script evaluates the held-out test set after training and saves plots and predictions.

Do not select hyperparameters by repeatedly examining the test scores. Keep this cell running until completion. The current trainer saves the best model but does **not** resume optimizer/scheduler state after a disconnect; rerunning starts training again. A disconnected run may have a checkpoint but no completed evaluation.

If CUDA runs out of memory, first restart the runtime and rerun setup while retaining these settings. If a smaller width is necessary, use a separate run directory, document the changed configuration, and do not label it width 16.

In [ ]:
assert torch.cuda.is_available(), 'GPU must be enabled.'
TRAIN_ARGS = ['--seed', 42, '--epochs', 60, '--batch-size', 1,
              '--gradient-accumulation', 4, '--learning-rate', 0.0002,
              '--weight-decay', 0.0001, '--patience', 12,
              '--num-workers', 2, '--base-channels', 16,
              '--group-norm-groups', 8, '--dropout', 0.30]
run('-m', 'btp3d.train', '--manifest', CACHED, '--output-dir', RUN, *TRAIN_ARGS)


## 10. Read results and demonstrate held-out-patient inference
The chosen patient comes from the fixed test split. This is a repeat inference demonstration, not an independent external validation sample. ROI inference requires a segmentation mask.

In [ ]:
metrics = json.loads((RUN / 'test_metrics.json').read_text())
assert metrics['device'].startswith('cuda') and metrics['n_patients'] == 56
checkpoint = torch.load(RUN / 'best_model.pt', map_location='cpu', weights_only=False)
assert checkpoint['model_config']['base_channels'] == 16
assert checkpoint['preprocess_config']['spatial_size'] == [96, 96, 96]
print(json.dumps(metrics, indent=2))
for name in ['loss_curves.png', 'confusion_matrix.png', 'roc_curve.png']:
    display(Image.open(RUN / name))
patient = manifest.loc[manifest.split == 'test'].iloc[0]
infer_args = []
for modality in ['flair', 't1', 't1ce', 't2', 'seg']:
    infer_args += ['--' + modality, patient[modality]]
run('-m', 'btp3d.infer', '--checkpoint', RUN / 'best_model.pt',
    *infer_args, '--output', RUN / 'example_prediction.json')


## 11. Additional experiment: five-fold GPU evaluation
Run after the main experiment if your GPU session budget permits. This trains five additional models. Each patient is tested once; a separate fold is used for validation. The existing script begins again from fold zero when rerun; it has no automatic fold-resume feature. The checkpoint folders on Drive survive, but an interrupted command may need a deliberate restart plan.

Keep this separate from the 32-cubed CPU cross-validation results.

In [ ]:
RUN_CV = False  # Change to True only when ready for five additional training runs.
if RUN_CV:
    assert torch.cuda.is_available()
    run('-m', 'btp3d.cross_validate', '--manifest', CACHED,
        '--output-dir', ARTIFACTS / 'cv5', *TRAIN_ARGS)
    print((ARTIFACTS / 'cv5' / 'cross_validation_metrics.json').read_text())


## 12. Additional experiment: whole-brain comparison
Uses the same patient splits and training settings. Brain cropping does not use the tumour mask. Compare its fixed-split results with the ROI fixed-split results; do not compare different evaluation schemes as if they were identical.

In [ ]:
RUN_WHOLE_BRAIN = False  # Change to True when ready for one additional model.
if RUN_WHOLE_BRAIN:
    BRAIN_CACHE = Path('/content/brain96_cache')
    run('-m', 'btp3d.preprocess', '--manifest', MANIFEST, '--output-dir', BRAIN_CACHE,
        '--spatial-size', 96, 96, 96, '--crop-mode', 'brain', '--margin', 0, '--workers', 2)
    run('-m', 'btp3d.train', '--manifest', BRAIN_CACHE / 'manifest_cached.csv',
        '--output-dir', ARTIFACTS / 'whole_brain_fixed', *TRAIN_ARGS)


## 13. Verify and save the completed run
Keep the notebook with its executed outputs. Drive already contains the saved results; this cell also creates a download ZIP. It excludes raw MRI and caches.

If the runtime resets: remount Drive, reinstall code, obtain the data again and rebuild `/content` caches before starting a new training command. Saved manifests contain runtime paths and cannot alone restore deleted cached tensors.

Training time depends on the assigned GPU and runtime limits. Measure the first run rather than assuming an exact duration. Five-fold evaluation requires five additional model fits; whole-brain comparison requires one more.

In [ ]:
required = ['best_model.pt', 'history.csv', 'test_predictions.csv', 'test_metrics.json',
            'loss_curves.png', 'confusion_matrix.png', 'roc_curve.png']
assert all((RUN / name).is_file() for name in required), 'Main experiment is incomplete.'
print('Completed main run:', RUN)
archive = shutil.make_archive(str(Path('/content') / f'BTP1_GPU_Results_{RUN_ID}'), 'zip', ARTIFACTS)
from google.colab import files
files.download(archive)
